Replay Batch B and prove that cumulative Silver and Gold business state remain unchanged.
The audit/bronze layers may record a new execution attempt.

In [0]:
dbutils.library.restartPython()

In [0]:
import sys
from pathlib import Path

project_root = Path.cwd().parent

sys.path.insert(0, str(project_root))

print("Project root:", project_root)
print("src exists:", (project_root / "src").is_dir())

In [0]:
from pyspark.sql import functions as F
from src.config import config
from src.pipeline import process_batch
from src.quality import check_idempotence_after_replay

#### Rebuild Batch B independently

In [0]:
source_df = spark.table(config.source_transactions)
split_date = "2024-05-15"
batch_b_df = source_df.filter(F.col("dateTime") >= F.lit(split_date).cast("timestamp"))

#### Persist pre-replay snapshots

In [0]:
silver_before_table = f"{config.catalog}.{config.schema}.silver_before_replay_snapshot"
gold_before_table = f"{config.catalog}.{config.schema}.gold_before_replay_snapshot"

(
    spark.read.table(config.silver_transactions)
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema","true")
    .saveAsTable(silver_before_table)
)
(
    spark.read.table(config.gold_daily_franchise_sales)
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema","true")
    .saveAsTable(gold_before_table)
)

#### Replay Batch B

In [0]:
process_batch(spark=spark, batch_df=batch_b_df, batch_id="batch_b", config=config)

#### Silver idempotence

In [0]:
silver_before = spark.read.table(silver_before_table)
silver_after = spark.read.table(config.silver_transactions)

check_idempotence_after_replay(
    before_df=silver_before, after_df=silver_after,
    key="transactionID", revenue_col="totalPrice", layer="Silver"
)


#### Gold idempotence

In [0]:
gold_before = spark.table(gold_before_table)
gold_after = spark.table(config.gold_daily_franchise_sales)

check_idempotence_after_replay(
    before_df=gold_before, after_df=gold_after,
    key=["franchiseID","transaction_date"], revenue_col="sum_revenue", layer="Gold"
)
